# Datos de FRED: descarga, preparación y panel trimestral

Este notebook reproduce paso a paso la preparación de datos de la investigación
"Condiciones macroeconómicas, política monetaria y riesgo de crédito".

## 1. ¿Qué es FRED?

FRED (Federal Reserve Economic Data) es la base de datos del Banco de la Reserva Federal
de St. Louis. Ofrece cientos de miles de series macroeconómicas de EE. UU. y otros países.

El ejemplo de la guía del profesor (sección 11) descarga cuatro series mensuales:

```python
# EJEMPLO DE LA GUÍA (no se ejecuta aquí; usa red)
from pandas_datareader import data as web
gdp = web.DataReader('GDP', 'fred', start='2000-01-01')
unrate = web.DataReader('UNRATE', 'fred', start='2000-01-01')
cpi = web.DataReader('CPIAUCSL', 'fred', start='2000-01-01')
ffr = web.DataReader('FEDFUNDS', 'fred', start='2000-01-01')
```

En este notebook **no usamos la red**: leemos las descargas crudas ya guardadas en `data/raw/`.
Así garantizamos reproducibilidad total (la spec 001 congela las descargas con fecha).

In [ ]:
# 2. Localizar la raíz del proyecto y definir RAW_DIR
import os
from pathlib import Path

# Subir directorios hasta encontrar pyproject.toml
ROOT = Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

RAW_DIR = Path(os.environ.get('RAW_DIR', ROOT / 'data' / 'raw'))
print(f'Raíz del proyecto: {ROOT}')
print(f'Directorio de descargas crudas: {RAW_DIR}')

In [ ]:
# 3. Cargar descargas crudas más recientes
import sys
sys.path.insert(0, str(ROOT))

from src.fred_io import load_latest_raw

try:
    raw, manifest = load_latest_raw(RAW_DIR)
    print(f'Series cargadas: {list(raw.keys())}')
    print(f'Manifiesto: {len(manifest)} filas')
    print(manifest[['series', 'download_date', 'source', 'n_rows', 'first_obs', 'last_obs']].to_string(index=False))
except FileNotFoundError as e:
    print(f'ERROR: {e}')
    print('Ejecuta: uv run python -m src.pipeline data --download')
    raise

In [ ]:
# 4. Reproducir el ejemplo de la guía (GDP, UNRATE, CPIAUCSL, FEDFUNDS)
#    combinando las descargas crudas, sin usar la red
import pandas as pd

# Extraer las 4 series del ejemplo
example_series = ['GDP', 'UNRATE', 'CPIAUCSL', 'FEDFUNDS']
dfs = []
for sid in example_series:
    df = raw[sid].copy()
    df.columns = ['date', sid]
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')
    dfs.append(df)

example_df = pd.concat(dfs, axis=1)
example_df = example_df.loc['2000-01-01':'2026-01-01']
print('Ejemplo de la guía (2000-01-01 a 2026-01-01):')
print(example_df.head())
print(example_df.tail())

## 5. Agregación trimestral, excepción 2025T4 y transformaciones

### Agregación trimestral (RF-8)
- Series mensuales (UNRATE, CPIAUCSL, FEDFUNDS, LOANS, USREC): **media de los meses del trimestre**.
- Series trimestrales (DRALACBS, GDPC1, GDP, etc.): **dato de fin de periodo** (último mes).
- Un trimestre es **completo** solo si tiene los 3 meses. Si falta alguno → NaN (RF-13).

### Excepción documentada 2025T4 (RF-11, RF-12)
- Octubre 2025 falta en CPIAUCSL y UNRATE (cierre del gobierno de EE. UU.).
- Si falta **solo octubre** → promedio de noviembre y diciembre (**RF-11**).
- Si octubre está presente → promedio de los 3 meses (**RF-12**).
- Esta excepción **solo** aplica a CPIAUCSL y UNRATE; en FEDFUNDS o LOANS sería error.

### Transformaciones (RF-9)
Calculadas sobre la **historia completa desde 1998** antes de recortar a 2000T1:
- Morosidad (×5): nivel, diferencia trimestral (pp), logit.
- PIB real/nominal, IPC, LOANS: variación interanual (100·Δln).
- UNRATE, FEDFUNDS: nivel y diferencia trimestral (pp).
- CORALACBS: nivel.
- USREC: 1 si algún mes del trimestre en recesión.

In [ ]:
# 6. Construir el panel completo con build_panel
from src.panel import build_panel

result = build_panel(raw)

print(f'Panel: {result.panel.shape[0]} trimestres × {result.panel.shape[1]} variables')
print(f'Rango: {result.panel.index[0]} a {result.panel.index[-1]}')
print(f'Último trimestre común (end): {result.end}')
print()
print('Primeras filas:')
print(result.panel.head())
print()
print('Últimas filas:')
print(result.panel.tail())